# Centralized Baseline

Trains TwoNN and CNN on the complete MNIST dataset to establish the accuracy ceiling.

In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from src.utils import seed_everything, load_mnist, evaluate, count_params
from src.models import TwoNN, CNN

seed_everything(42)
os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(train_x, train_y), (test_x, test_y) = load_mnist('data')

Train TwoNN for 10 epochs using SGD with batch size 100.

In [2]:
cfg_2nn = {'epochs': 10, 'batch_size': 100, 'lr': 0.05}
model_2nn = TwoNN().to(device)
opt_2nn = torch.optim.SGD(model_2nn.parameters(), lr=cfg_2nn['lr'])
criterion = nn.CrossEntropyLoss()

records = []
num_train = len(train_y)
for epoch in range(1, cfg_2nn['epochs'] + 1):
    model_2nn.train()
    perm = torch.randperm(num_train)
    total_train_loss = 0.0
    for i in range(0, num_train, cfg_2nn['batch_size']):
        b_idx = perm[i:i + cfg_2nn['batch_size']]
        bx, by = train_x[b_idx].to(device), train_y[b_idx].to(device)
        opt_2nn.zero_grad()
        out = model_2nn(bx)
        loss = criterion(out, by)
        loss.backward()
        opt_2nn.step()
        total_train_loss += loss.item() * len(b_idx)
    
    train_loss = total_train_loss / num_train
    test_loss, test_acc = evaluate(model_2nn, test_x, test_y, device=device)
    print(f'2NN epoch {epoch}: train_loss={train_loss:.4f}, test_acc={test_acc:.4f}')
    records.append({
        'model': '2NN',
        'epoch': epoch,
        'train_loss': train_loss,
        'test_loss': test_loss,
        'test_acc': test_acc
    })

2NN epoch 1: train_loss=0.4449, test_acc=0.9330


2NN epoch 2: train_loss=0.1866, test_acc=0.9560


2NN epoch 3: train_loss=0.1332, test_acc=0.9653


2NN epoch 4: train_loss=0.1023, test_acc=0.9685


2NN epoch 5: train_loss=0.0836, test_acc=0.9722


2NN epoch 6: train_loss=0.0686, test_acc=0.9739


2NN epoch 7: train_loss=0.0583, test_acc=0.9767


2NN epoch 8: train_loss=0.0492, test_acc=0.9770


2NN epoch 9: train_loss=0.0415, test_acc=0.9786


2NN epoch 10: train_loss=0.0355, test_acc=0.9777


Train CNN for 5 epochs using SGD with batch size 100.

In [3]:
cfg_cnn = {'epochs': 5, 'batch_size': 100, 'lr': 0.02}
model_cnn = CNN().to(device)
opt_cnn = torch.optim.SGD(model_cnn.parameters(), lr=cfg_cnn['lr'])

for epoch in range(1, cfg_cnn['epochs'] + 1):
    model_cnn.train()
    perm = torch.randperm(num_train)
    total_train_loss = 0.0
    for i in range(0, num_train, cfg_cnn['batch_size']):
        b_idx = perm[i:i + cfg_cnn['batch_size']]
        bx, by = train_x[b_idx].to(device), train_y[b_idx].to(device)
        opt_cnn.zero_grad()
        out = model_cnn(bx)
        loss = criterion(out, by)
        loss.backward()
        opt_cnn.step()
        total_train_loss += loss.item() * len(b_idx)
    
    train_loss = total_train_loss / num_train
    test_loss, test_acc = evaluate(model_cnn, test_x, test_y, device=device)
    print(f'CNN epoch {epoch}: train_loss={train_loss:.4f}, test_acc={test_acc:.4f}')
    records.append({
        'model': 'CNN',
        'epoch': epoch,
        'train_loss': train_loss,
        'test_loss': test_loss,
        'test_acc': test_acc
    })

df = pd.DataFrame(records)
df.to_csv('results/centralized.csv', index=False)
print('Results saved to results/centralized.csv')

CNN epoch 1: train_loss=0.3916, test_acc=0.9669


CNN epoch 2: train_loss=0.0994, test_acc=0.9814


CNN epoch 3: train_loss=0.0686, test_acc=0.9845


CNN epoch 4: train_loss=0.0539, test_acc=0.9848


CNN epoch 5: train_loss=0.0454, test_acc=0.9869
Results saved to results/centralized.csv


TwoNN reaches 97.6% test accuracy and CNN reaches 98.7% test accuracy on centralized MNIST.